# 电商落地页转化率 A/B Test 分析

> 使用双侧两比例 $z$ 检验与 95% 置信区间，评估新落地页是否带来可验证的转化率变化。

## 项目背景与分析目标

某电商平台希望用新版落地页替换现有页面，目标是提升用户完成目标行为的转化率。上线前需要通过随机对照实验判断观察到的变化是否足以支持产品决策，而不能只比较两个样本均值。

### 实验设计

| 项目 | 定义 |
|---|---|
| 对照组 | `control`，展示 `old_page` |
| 实验组 | `treatment`，展示 `new_page` |
| 分析单位 | 用户；每位用户只保留首次有效实验记录 |
| 核心指标 | 转化率，即 `converted` 的均值 |
| 显著性水平 | $\alpha=0.05$ |
| 检验方法 | 双侧两比例 $z$ 检验 |
| 效果区间 | 新页面减旧页面的转化率差及其 95% 置信区间 |

### 核心问题

1. 实验分组与实际展示页面是否一致，数据是否满足分析要求？
2. 新旧页面的总体转化率及绝对、相对变化分别是多少？
3. 观察到的差异是否达到统计显著，真实效果的合理范围有多大？
4. 现有证据是否支持新页面全量上线？

### 决策口径

本分析关注双侧差异：当 $p<0.05$ 且效果方向与业务目标一致时，才认为数据提供了支持上线的统计证据。若结果不显著，只能说明当前证据不足，不能证明两个页面完全等效。

## 1. 数据导入与字段说明

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ab_data.csv"

print("项目目录:", PROJECT_ROOT.resolve())
print("数据文件", DATA_PATH.resolve())
print("文件存在:", DATA_PATH.exists())

ab_data = pd.read_csv(DATA_PATH)
print("数据形状 :", ab_data.shape)
display(ab_data.head())
display(ab_data.dtypes.to_frame("数据类型 "))

项目目录: C:\Users\Administrator\Desktop\learning materials\个人项目\电商落地页转化率 A-B Test 分析
数据文件 C:\Users\Administrator\Desktop\learning materials\个人项目\电商落地页转化率 A-B Test 分析\data\raw\ab_data.csv
文件存在: True
数据形状 : (294478, 5)


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


,数据类型
user_id,int64
timestamp,str
group,str
landing_page,str
converted,int64


user_id：一位用户的标识。<br>
timestamp：用户进入实验的时间。<br>
group：实验分组，control 是对照组，treatment 是实验组。<br>
landing_page：用户实际看到的页面，old_page 或 new_page。<br>
converted：最终是否完成目标转化，1 为转化，0 为未转化。<br>

## 2. 数据质量检查与清洗

检查实验分流、页面映射、转化结果、缺失值和重复用户。有效记录应满足 `control → old_page` 或 `treatment → new_page`；同一用户重复进入实验时，仅保留时间最早的有效记录，避免重复计权。

In [3]:
print("实验分组:", ab_data["group"].value_counts().to_dict())
print("落地页版本:", ab_data["landing_page"].value_counts().to_dict())

group_page_table = pd.crosstab(ab_data["group"], ab_data["landing_page"])
display(group_page_table)

print("转化字段取值:", sorted(ab_data["converted"].dropna().unique()))
print("重复用户 ID 数:", ab_data["user_id"].duplicated().sum())
print("缺失值:")
display(ab_data.isna().sum().to_frame("缺失值"))

实验分组: {'treatment': 147276, 'control': 147202}
落地页版本: {'old_page': 147239, 'new_page': 147239}


landing_page,new_page,old_page
group,,
control,1928,145274
treatment,145311,1965


转化字段取值: [np.int64(0), np.int64(1)]
重复用户 ID 数: 3894
缺失值:


,缺失值
user_id,0
timestamp,0
group,0
landing_page,0
converted,0


##### 结果分析：
分流接近 50/50：control 147,202，treatment 147,276。<br>
没有缺失值，converted 只有 0/1，基础质量合格。<br>
但有 3,893 条分组与页面不匹配：<br>
其中<br>
control + new_page：1,928<br>
treatment + old_page：1,965<br>
有 3,894 条重复用户 ID。<br>
<br>下一步目标：
需要去除在control + new_page和treatment + old_page这两种异常组合中的用户数据，以及重复的用户数据。

In [4]:
# 只保留control -> old_page，treatment -> new_page这两种组合形式的数据
save = (
    ((ab_data["group"] == "control") & (ab_data["landing_page"] == "old_page"))
    | ((ab_data["group"] == "treatment") & (ab_data["landing_page"] == "new_page") )
)

ab_clean = ab_data.loc[save].copy()

ab_clean["timestamp"] = pd.to_datetime(ab_clean["timestamp"])

ab_clean = (
    ab_clean
    .sort_values("timestamp")
    .drop_duplicates(subset="user_id", keep="first")
    .reset_index(drop=True)
)  ##删除重复用户，保留第一次访问记录。

print("原始记录数:", len(ab_data))
print("清洗后记录数:", len(ab_clean))
print("移除记录数:", len(ab_data) - len(ab_clean))

print("\n清洗后分组与页面:")
display(pd.crosstab(ab_clean["group"], ab_clean["landing_page"]))

print(
    "清洗后重复用户数:",
    ab_clean["user_id"].duplicated().sum(),
)

print(
    "清洗后缺失值总数:",
    ab_clean.isna().sum().sum(),
)

assert ab_clean["user_id"].is_unique
assert ab_clean.isna().sum().sum() == 0
print("数据清洗完成")

ab_clean.to_csv(
    PROJECT_ROOT / "data" / "processed" / "ab_clean.csv",
    index=False,
    encoding="utf-8-sig",
)

原始记录数: 294478
清洗后记录数: 290584
移除记录数: 3894

清洗后分组与页面:


landing_page,new_page,old_page
group,,
control,0,145274
treatment,145310,0


清洗后重复用户数: 0
清洗后缺失值总数: 0
数据清洗完成


## 3. 总体转化率与效果量

In [5]:
conversion_summary = (
    ab_clean.groupby("group")["converted"]
    .agg(sample_size="count", conversions="sum", conversion_rate="mean")
)

control_rate = conversion_summary.loc["control", "conversion_rate"]
treatment_rate = conversion_summary.loc["treatment", "conversion_rate"]

absolute_lift = treatment_rate - control_rate
relative_lift = absolute_lift / control_rate

summary_display = conversion_summary.copy()

summary_display["conversion_rate"] = (
    summary_display["conversion_rate"].map("{:.4%}".format)
)

summary_display["absolute_lift"] = ""
summary_display["relative_lift"] = ""

summary_display.loc["treatment", "absolute_lift"] = f"{absolute_lift:.4%}"
summary_display.loc["treatment", "relative_lift"] = f"{relative_lift:.2%}"

display(summary_display)

,sample_size,conversions,conversion_rate,absolute_lift,relative_lift
group,,,,,
control,145274,17489,12.0386%,,
treatment,145310,17264,11.8808%,-0.1578%,-1.31%


## 4. 假设检验（两比例 z 检验）

#### 假设
- 原假设 $H_0$：新旧落地页的真实转化率相同。

$$
H_0: p_{\text{treatment}} = p_{\text{control}}
$$

- 备择假设 $H_1$：新旧落地页的真实转化率不同。

$$
H_1: p_{\text{treatment}} \ne p_{\text{control}}
$$

本项目的 `converted` 是 0/1 二元变量，目标是比较实验组与对照组两个独立大样本的转化率。两组样本量均约 14.5 万，转化人数均超过 1.7 万，满足大样本比例正态近似条件，因此采用双侧两比例 z 检验。

两比例 $z$ 检验统计量：

$$
z =
\frac{
\hat{p}_{\text{treatment}} -
\hat{p}_{\text{control}}
}{
\sqrt{
\hat{p}(1 - \hat{p})
\left(
\frac{1}{n_{\text{treatment}}}
+
\frac{1}{n_{\text{control}}}
\right)
}
}
$$ <br>
- $\hat{p}_{\text{treatment}}$：实验组样本转化率
- $\hat{p}_{\text{control}}$：对照组样本转化率
- $n_{\text{treatment}}$：实验组样本量
- $n_{\text{control}}$：对照组样本量
- $\hat{p}$：两组在原假设下的合并转化率

In [6]:
from statsmodels.stats.proportion import proportions_ztest ##比较两个比例
control_conversions = conversion_summary.loc["control", "conversions"]
treatment_conversions = conversion_summary.loc["treatment", "conversions"]

control_size = conversion_summary.loc["control", "sample_size"]
treatment_size = conversion_summary.loc["treatment", "sample_size"]

z_stat, p_value = proportions_ztest(
    count=[treatment_conversions, control_conversions],
    nobs=[treatment_size, control_size],
    alternative="two-sided",
)

print(f"Z 统计量: {z_stat:.4f}")
print(f"P 值: {p_value:.6f}")

alpha = 0.05

if p_value > alpha:
    print(f"结论：p‑value = {p_value:.4f} > α={alpha}，不能拒绝 H0，没有足够证据认为新旧页面的转化率存在显著差异。")
else:
    print(f"结论：p‑value = {p_value:.4f} ≤ α={alpha}，拒绝H0，新旧页面转化率存在显著差异。")

Z 统计量: -1.3109
P 值: 0.189883
结论：p‑value = 0.1899 > α=0.05，不能拒绝 H0，没有足够证据认为新旧页面的转化率存在显著差异。


## 5. 效果量的 95% 置信区间

$z$ 检验表明当前差异未达到统计显著性。下面计算新页面相对旧页面转化率绝对变化的 95% 置信区间，量化真实差异可能的范围。

转化率差异的标准误：

$$
SE_{\Delta} =
\sqrt{
\frac{
\hat{p}_{\text{treatment}}
\left(1-\hat{p}_{\text{treatment}}\right)
}{
n_{\text{treatment}}
}
+
\frac{
\hat{p}_{\text{control}}
\left(1-\hat{p}_{\text{control}}\right)
}{
n_{\text{control}}
}
}
$$

95% 置信区间：

$$
CI_{95\%} =
\hat{\Delta}
\pm
1.96 \times SE_{\Delta}
$$
- $\hat{\Delta}$：样本中的转化率绝对变化，即 `absolute_lift`
- $SE_{\Delta}$：两个样本转化率之差的标准误
- $1.96$：95% 置信水平对应的标准正态临界值

In [7]:
confidence_level = 0.95
z_critical = 1.96

standard_error_diff = np.sqrt(
    treatment_rate * (1 - treatment_rate) / treatment_size
    + control_rate * (1 - control_rate) / control_size
)

ci_lower = absolute_lift - z_critical * standard_error_diff
ci_upper = absolute_lift + z_critical * standard_error_diff

print(f"转化率绝对变化: {absolute_lift:.4%}")
print(
    f"{confidence_level:.0%} 置信区间: "
    f"[{ci_lower:.4%}, {ci_upper:.4%}]"
)

转化率绝对变化: -0.1578%
95% 置信区间: [-0.3938%, 0.0781%]


#### 结果解读

- 点估计：新页面转化率比旧页面低 0.1578 个百分点。
- 95% 置信区间：$[-0.3938\%,\ 0.0781\%]$。
- 该区间包含 $0$，因此无法排除新旧页面真实转化率没有差异的可能。
- 该结果与两比例 $z$ 检验中 $p$ 值大于 $0.05$ 的结论一致：当前没有足够证据表明新旧页面的真实转化率存在统计显著差异。

## 6. 业务结论与建议

### 决策结论

**暂不建议以“提升转化率”为理由全量上线新页面。**

- 旧页面转化率为 **12.0386%**，新页面为 **11.8808%**。
- 新页面的绝对变化为 **-0.1578 个百分点**，相对变化为 **-1.31%**。
- 双侧两比例 $z$ 检验得到 $p=0.1899$，未达到 $\alpha=0.05$ 的显著性标准。
- 95% 置信区间为 **[-0.3938%, 0.0781%]**，跨越 0；真实效果可能是小幅下降、无差异或小幅提升。

### 后续行动

1. 维持旧页面作为当前默认方案，避免把未验证的改版收益当作确定结果。
2. 结合页面点击、表单流失等过程指标定位新版问题，形成有明确机制假设的新方案。
3. 下一轮实验开始前预先定义主要指标、最小可检测效应、样本量和停止规则，避免依据中途结果提前结束实验。
4. 若业务关心“新页面不比旧页面差”，应另行设定可接受的劣效界值并采用非劣效检验；本次双侧检验不能证明两版页面等效。

### 分析边界

数据不包含收入、利润、设备、渠道或用户分层信息，因此本项目只评价总体转化率，不推断收入影响，也不解释不同用户群体中的异质性效果。

## 项目分析流程与关键发现

```mermaid
flowchart TD
    A["原始实验数据<br/>294,478 条记录"] --> B["数据质量检查"]

    B --> B1["无缺失值<br/>converted 仅含 0 和 1"]
    B --> B2["发现 3,893 条分组与页面不匹配记录"]
    B --> B3["发现重复用户记录"]

    B1 --> C["数据清洗"]
    B2 --> C
    B3 --> C

    C --> C1["仅保留 control - old_page<br/>treatment - new_page"]
    C1 --> C2["按时间保留每位用户的首次记录"]
    C2 --> D["清洗后样本<br/>290,584 位唯一用户"]

    D --> E["计算两组转化率"]
    E --> E1["旧页面<br/>145,274 人<br/>转化率：12.0386%"]
    E --> E2["新页面<br/>145,310 人<br/>转化率：11.8808%"]

    E1 --> F["估计新页面相对旧页面的效果"]
    E2 --> F
    F --> F1["绝对变化：-0.1578 个百分点"]
    F --> F2["相对变化：-1.31%"]

    F1 --> G["双侧两比例 z 检验<br/>p-value：0.1899，大于 0.05"]
    F2 --> G
    G --> G1["不能拒绝 H0<br/>未发现统计显著差异"]

    G1 --> H["计算 95% 置信区间<br/>[-0.3938%, 0.0781%]"]
    H --> H1["区间包含 0<br/>真实效果可能为下降、无差异或小幅提升"]

    H1 --> I["业务建议"]
    I --> I1["不建议以提升转化率为理由<br/>直接全量上线新页面"]
    I --> I2["保留旧页面，优化新页面后<br/>重新开展 A/B Test"]
    I --> I3["使用 Tableau 展示<br/>转化率、差异、p 值和置信区间"]

## 7. 交互式 Tableau Dashboard

看板支持点击新旧页面转化率柱形，联动筛选对应的每日转化率趋势。

<iframe
    src="https://public.tableau.com/views/ABtest_17871506401860/1?:showVizHome=no"
    width="100%"
    height="850"
    frameborder="0"
    title="电商落地页 A/B Test 交互式 Tableau Dashboard">
</iframe>

> GitHub 会出于安全原因过滤 Notebook 中的 `iframe`。若当前页面未显示看板，请访问 [Tableau Public 交互式 Dashboard](https://public.tableau.com/views/ABtest_17871506401860/1?:showVizHome=no)。